# Lab: Enterprise Graph RAG with Neo4j
In this lab, we build a production-ready Graph RAG pipeline. We will extract entities and relationships dynamically from a PDF using an LLM, load them into a Neo4j graph database, and use Cypher queries to traverse the graph and answer user questions.

### Step 1: Install Dependencies

In [ ]:
!pip install neo4j requests PyPDF2 yfiles-jupyter-graphs-for-neo4j


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Import Libraries & Setup Credentials

In [1]:
import os
import json
import requests
import PyPDF2
from neo4j import GraphDatabase

In [ ]:
# --- LLM Configuration ---
OPENROUTER_API_KEY = "your-api-key"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
TEXT_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

In [ ]:
# --- Neo4j Cloud Database Configuration ---
NEO4J_URI = "YOUR-NEO4J_URI"
NEO4J_USER = "YOUR-NEO4J_USER"
NEO4J_PASSWORD = "YOUR-NEO4J_PASSWORD"

# Open a persistent connection to the database
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Success: Connected to Neo4j Database!")

Success: Connected to Neo4j Database!


### Step 3: Extract PDF Text

In [ ]:
# We will download a famous AI research paper on Transformers
PDF_URL = "https://arxiv.org/pdf/1706.03762.pdf"

os.makedirs("data", exist_ok=True)
PDF_PATH = os.path.join("data", "document.pdf")

print("Downloading PDF...")
response = requests.get(PDF_URL)
with open(PDF_PATH, "wb") as f:
    f.write(response.content)

# Extract text from the first page
with open(PDF_PATH, "rb") as file:
    reader = PyPDF2.PdfReader(file)
    sample_text = reader.pages[0].extract_text()

# We only use the first 1500 characters to keep API processing fast and free
sample_text = sample_text[:1500].strip()
print(f"Success! Extracted {len(sample_text)} characters ready for AI processing.")

Success! Extracted 1500 characters ready for processing.


### Step 4: Domain-Agnostic Knowledge Extraction

In [ ]:
def extract_graph_elements(text):
    prompt = f"""
    You are an expert knowledge graph builder. Read the text below and extract all key concepts and their relationships.

    Text:
    {text}

    CRITICAL INSTRUCTIONS:
    Output ONLY a valid JSON list of objects with keys "source", "relation", and "target".
    Example format:
    [
      {{"source": "Entity_A", "relation": "RELATES_TO", "target": "Entity_B"}}
    ]
    Return ONLY pure JSON. Do not add explanations.
    """
    
    payload = {
        "model": TEXT_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0 # Keeps the AI strictly factual
    }
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    
    try:
        print("Extracting relationships... (This may take a minute)")
        resp = requests.post(OPENROUTER_URL, headers=headers, json=payload)
        resp.raise_for_status()
        raw_json = resp.json()["choices"][0]["message"]["content"].strip()
        
        # Strip out markdown formatting if the AI added it
        if raw_json.startswith("```"):
            raw_json = raw_json.split("\n", 1)[1].rsplit("```", 1)[0].strip()
            
        return json.loads(raw_json)
    except Exception as e:
        print(f"Extraction Error: {e}")
        return []

extracted_relationships = extract_graph_elements(sample_text)
print(f"\nExtracted {len(extracted_relationships)} relationships from the text.")
print(json.dumps(extracted_relationships[:2], indent=2)) # Preview first 2

Extracting relationships... (This may take a minute)

Extracted 36 relationships from the text.
[
  {
    "source": "Attention Is All You Need",
    "relation": "HAS_AUTHOR",
    "target": "Ashish Vaswani"
  },
  {
    "source": "Attention Is All You Need",
    "relation": "HAS_AUTHOR",
    "target": "Noam Shazeer"
  },
  {
    "source": "Attention Is All You Need",
    "relation": "HAS_AUTHOR",
    "target": "Niki Parmar"
  }
]


### Step 5: Ingest Data into Neo4j
We use a Cypher query to insert the extracted nodes and edges directly into the database. 
*(Note: Once this completes, open your Neo4j Browser in the Aura Console and run `MATCH (n) RETURN n` to see your interactive graph!)*

In [ ]:
def insert_into_neo4j(tx, relationships):
    for rel in relationships:
        source = rel["source"].strip()
        target = rel["target"].strip()
        
        # Neo4j relation names must be uppercase with underscores (e.g., HAS_AUTHOR)
        relation_type = rel["relation"].strip().replace(" ", "_").replace("-", "_").upper()
        
        # Cypher query using MERGE so we don't create duplicate bubbles
        query = f"""
        MERGE (s:Concept {{name: $source}})
        MERGE (t:Concept {{name: $target}})
        MERGE (s)-[:{relation_type}]->(t)
        """
        tx.run(query, source=source, target=target)

# Connect to the DB and load the data
with driver.session() as session:
    # Clear out the database first for a clean lab environment
    session.run("MATCH (n) DETACH DELETE n")
    
    session.execute_write(insert_into_neo4j, extracted_relationships)
    print("Success: Knowledge Graph loaded into Neo4j!")

Database cleared for fresh ingestion.
Knowledge Graph successfully loaded into Neo4j!


### Step 6: Dynamic Traversal via Cypher

In [ ]:
def find_node_in_db(question):
    """Searches the database to see if a word in the question matches a known node."""
    with driver.session() as session:
        result = session.run("MATCH (c:Concept) RETURN c.name AS name")
        for record in result:
            node_name = record["name"]
            if node_name.lower() in question.lower():
                return node_name
    return None

In [ ]:
def fetch_graph_context(entity_name):
    """Pulls all direct relationships connected to our target node."""
    query = """
    MATCH (n:Concept {name: $name})-[r]-(m:Concept)
    RETURN n.name AS source, type(r) AS relation, m.name AS target
    """
    facts = []
    with driver.session() as session:
        result = session.run(query, name=entity_name)
        for record in result:
            facts.append(f"{record['source']} --[{record['relation']}]--> {record['target']}")
    return facts

### Step 7: The Neo4j Graph RAG Engine

In [ ]:
def execute_neo4j_rag(question):
    # Step 1: Detect if any entity from the database is mentioned in the question
    target_entity = find_node_in_db(question)
    if not target_entity:
        return "Could not find any known concepts from the database in your question."
        
    print(f"[System Log] Auto-detected focus concept: '{target_entity}'")
    
    # Step 2: Retrieve connected relationship paths from Neo4j
    retrieved_facts = fetch_graph_context(target_entity)
    if not retrieved_facts:
        return f"Found '{target_entity}' in Neo4j, but no relationships were connected to it."
        
    facts_block = "\n".join([f"- {f}" for f in retrieved_facts])
    
    # Step 3: Build a prompt that forces the LLM to rely strictly on graph facts
    prompt = f"""
    You are an expert AI research assistant using a Neo4j Knowledge Graph.
    Answer the question using ONLY the connected relationship paths provided below.

    Graph Relationships:
    {facts_block}

    Question: {question}

    CRITICAL INSTRUCTIONS:
    Output your response in EXACTLY two sections as shown below.

    --- FINAL ANSWER ---
    [Provide a direct, simple, 1-sentence answer.]

    --- AI TRACING & EXPLAINABILITY ---
    [Explain step-by-step how the answer was derived from the Neo4j graph. Use an objective, third-person perspective.]
    """
    
    payload = {
        "model": TEXT_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0
    }
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    
    # Step 4: Send prompt to LLM and return the structured answer
    try:
        resp = requests.post(OPENROUTER_URL, headers=headers, json=payload)
        resp.raise_for_status()
        
        response_json = resp.json()
        if "choices" in response_json:
            return response_json["choices"][0]["message"]["content"].strip()
        return f"OpenRouter API Error: {response_json}"
        
    except Exception as e:
        return f"Error executing Graph RAG: {e}"

### Step 8: Test the Database RAG

In [7]:
query = "What mechanism does the Transformer architecture rely on?"
print(execute_neo4j_rag(query))

[System Log] Auto-detected focus concept: 'Transformer'
--- FINAL ANSWER ---
The Transformer architecture relies on attention mechanisms.

--- AI TRACING & EXPLAINABILITY ---
The answer was derived by examining the provided Neo4j graph relationships for the Transformer node. A direct relationship labeled BASED_ON connects the Transformer node to the "attention mechanisms" node, explicitly indicating that the architecture is founded on this mechanism. No other mechanism-related relationships (such as PROPOSES, HAS_COMPONENT, or DISPENSES_WITH) describe a foundational reliance; they describe components, training details, or removed elements like recurrence and convolutions. Therefore, the graph supports the conclusion that attention mechanisms are the core mechanism the Transformer relies on.


In [8]:
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget

# 'driver' is the connection we already opened in Step 2!
widget = Neo4jGraphWidget(driver)

# Run the exact same query you would run in the Aura console
widget.show_cypher("MATCH (n)-[r]->(m) RETURN n, r, m")

GraphWidget(layout=Layout(height='800px', width='100%'))